In [2]:
import pandas as pd
from pathlib import Path
import numpy as np
import geopandas as gpd
from shapely.geometry import Point

## Potholes

In [3]:
# load csvs and gpkgs
df_2016 = pd.read_csv("datasets/pothole_fixes/potholes_2016.csv")
df_2017 = pd.read_csv("datasets/pothole_fixes/potholes_2017.csv")
df_2018 = pd.read_csv("datasets/pothole_fixes/potholes_2018.csv")
df_2019 = pd.read_csv("datasets/pothole_fixes/potholes_2019.csv")
df_2020 = pd.read_csv("datasets/pothole_fixes/potholes_2020.csv")
gdf_2021 = gpd.read_file("datasets/pothole_fixes/potholes_2021.gpkg")
gdf_2022 = gpd.read_file("datasets/pothole_fixes/potholes_2022.gpkg")
gdf_2023 = gpd.read_file("datasets/pothole_fixes/potholes_2023.gpkg")
gdf_2024 = gpd.read_file("datasets/pothole_fixes/potholes_2024.gpkg")
gdf_2025 = gpd.read_file("datasets/pothole_fixes/potholes_2025.gpkg")

In [4]:
# convert csvs to gpkg
def csv_to_gdf(df, lon_col='Longitude', lat_col='Latitude'):
    geometry = [Point(lon, lat) for lon, lat in zip(df[lon_col], df[lat_col])]
    return gpd.GeoDataFrame(df, geometry=geometry, crs='EPSG:4326')

gdf_2016 = csv_to_gdf(df_2016)
gdf_2017 = csv_to_gdf(df_2017)
gdf_2018 = csv_to_gdf(df_2018)
gdf_2019 = csv_to_gdf(df_2019)
gdf_2020 = csv_to_gdf(df_2020)

In [5]:
# add source year to gdfs
gdf_2016["source_year"] = 2016
gdf_2017["source_year"] = 2017
gdf_2018["source_year"] = 2018
gdf_2019["source_year"] = 2019
gdf_2020["source_year"] = 2020
gdf_2021["source_year"] = 2021
gdf_2022["source_year"] = 2022
gdf_2023["source_year"] = 2023
gdf_2024["source_year"] = 2024
gdf_2025["source_year"] = 2025

In [6]:
# concatenate all gdfs and drop unnecessary columns
gdfs = [gdf_2016, gdf_2017, gdf_2018, gdf_2019, gdf_2020, gdf_2021, gdf_2022, gdf_2023, gdf_2024, gdf_2025]
gdfs = [gdf.to_crs("EPSG:4326") for gdf in gdfs]
potholes = gpd.GeoDataFrame(pd.concat(gdfs, ignore_index=True))
# just keep DateJour
potholes['DateJour'] = potholes['DateJour'].fillna(potholes['Date'].astype(str).str[:10])
potholes = potholes.drop(columns=['Appareil', 'Véhicule', 'Date', 'DateHeure'])
potholes['Latitude'] = potholes['Latitude'].fillna(potholes.geometry.y)
potholes['Longitude'] = potholes['Longitude'].fillna(potholes.geometry.x)
potholes['Date'] = pd.to_datetime(potholes['DateJour'])
potholes = potholes.drop(columns=['DateJour'])
# export file
potholes.to_file("datasets/potholes_cleaned.gpkg", driver="GPKG")

KeyboardInterrupt: 

## Road Assets

In [7]:
road_assets = gpd.read_file("datasets/road_assets/voirie_actif.geojson")

In [8]:
# drop unnecessary columns
keep_cols = [
    'ID_VOI_VOIRIE_AGR',
    'CATEGORIECHAUSSEE_REF',
    'DATECONSTRUCTION',
    'DATERESURFACAGE',
    'MATERIAUCHAUSSEE_REF',
    'TYPEFONDATION_REF',
    'UTILISATION_REF',
    'geometry'
]

road_assets = road_assets[keep_cols]
print(road_assets.info())

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 249348 entries, 0 to 249347
Data columns (total 8 columns):
 #   Column                 Non-Null Count   Dtype   
---  ------                 --------------   -----   
 0   ID_VOI_VOIRIE_AGR      249348 non-null  int32   
 1   CATEGORIECHAUSSEE_REF  63813 non-null   str     
 2   DATECONSTRUCTION       106682 non-null  str     
 3   DATERESURFACAGE        19497 non-null   str     
 4   MATERIAUCHAUSSEE_REF   63813 non-null   str     
 5   TYPEFONDATION_REF      65913 non-null   str     
 6   UTILISATION_REF        155316 non-null  str     
 7   geometry               249348 non-null  geometry
dtypes: geometry(1), int32(1), str(6)
memory usage: 14.3 MB
None


In [9]:
# Check how many are vehicle roads
print(road_assets['UTILISATION_REF'].value_counts())

# Filter to vehicle-related usage only
vehicle_usage = ['Véhicule', 'Véhicule stationnement']
road_assets_vehicles = road_assets[road_assets['UTILISATION_REF'].isin(vehicle_usage)]

print(f"Before: {len(road_assets)}")
print(f"After: {len(road_assets_vehicles)}")

UTILISATION_REF
Piéton                    69698
Véhicule                  56862
Non applicable            21298
Voie cyclable              5930
Traverse de piétons         865
Passage à niveau            494
Aménagement                  73
Véhicule stationnement       63
Ruelle verte                 27
Rue verte                     5
Inconnu                       1
Name: count, dtype: int64
Before: 249348
After: 56925


In [13]:
# Convert dates to date
# Convert to datetime (just need the first 8 characters for date)
road_assets['DATECONSTRUCTION'] = pd.to_datetime(
    road_assets['DATECONSTRUCTION'].str[:8],
    format='%Y%m%d',
    errors='coerce'
)

road_assets['DATERESURFACAGE'] = pd.to_datetime(
    road_assets['DATERESURFACAGE'].str[:8],
    format='%Y%m%d',
    errors='coerce'
)

AttributeError: Can only use .str accessor with string values, not datetime64

In [15]:
road_assets = road_assets[road_assets['UTILISATION_REF'] == 'Véhicule']

print(len(road_assets))
print(road_assets['DATECONSTRUCTION'].notna().sum())
print(road_assets['DATERESURFACAGE'].notna().sum())

56862
56860
7030


In [16]:
# For roads with no resurfacing, use construction date as the "last surface date"
road_assets['LAST_SURFACE_DATE'] = road_assets['DATERESURFACAGE'].fillna(road_assets['DATECONSTRUCTION'])

In [20]:
# Print to new file
road_assets.to_file("datasets/road_assets_cleaned.gpkg", driver="GPKG")

## Road Condition

In [25]:
rc_2010 = pd.read_csv("datasets/road_condition/auscultation-chaussees-2010.csv")
rc_2015 = pd.read_csv("datasets/road_condition/auscultation-chaussees-2015.csv")
rc_2018 = pd.read_csv("datasets/road_condition/auscultation-chaussees-2018-arteriel.csv")
rc_2020 = pd.read_csv("datasets/road_condition/auscultation-chaussees-2020-arteriel.csv")
rc_2022 = pd.read_csv("datasets/road_condition/auscultation-chaussees-2022-local.csv")
rc_2024 = pd.read_csv("datasets/road_condition/auscultation-chaussee-2024.csv")

In [27]:
rc_2010['source_year'] = 2010
rc_2015['source_year'] = 2015
rc_2018['source_year'] = 2018
rc_2020['source_year'] = 2020
rc_2022['source_year'] = 2022
rc_2024['source_year'] = 2024

road_condition = pd.concat([rc_2010, rc_2015, rc_2018, rc_2020, rc_2022, rc_2024], ignore_index=True)

In [31]:
keep_cols = ['ID_TRC', 'Rue', 'De', 'A', 'Longueur', 'Arrondissement', 'DateReleve', 'Indice_PCI', 'Indice_IRI']
road_condition = road_condition[keep_cols]

# Convert date
road_condition['DateReleve'] = pd.to_datetime(road_condition['DateReleve'], format='mixed')

print(road_condition.info())

<class 'pandas.DataFrame'>
RangeIndex: 121163 entries, 0 to 121162
Data columns (total 9 columns):
 #   Column          Non-Null Count   Dtype         
---  ------          --------------   -----         
 0   ID_TRC          121163 non-null  int64         
 1   Rue             121163 non-null  str           
 2   De              120871 non-null  str           
 3   A               120759 non-null  str           
 4   Longueur        121163 non-null  float64       
 5   Arrondissement  121163 non-null  str           
 6   DateReleve      121163 non-null  datetime64[us]
 7   Indice_PCI      46873 non-null   float64       
 8   Indice_IRI      44623 non-null   float64       
dtypes: datetime64[us](1), float64(3), int64(1), str(4)
memory usage: 8.3 MB
None


In [34]:
print(road_condition.groupby(road_condition['DateReleve'].dt.year)['Indice_PCI'].apply(lambda x: x.notna().sum()))

DateReleve
2009        0
2010        0
2011        0
2015        0
2018        0
2020    13876
2022    15821
2024    17176
Name: Indice_PCI, dtype: int64


In [35]:
road_condition.to_csv("datasets/road_condition_cleaned.csv", index=False)

print(f"Saved {len(road_condition)} records")

Saved 121163 records


## Traffic

In [38]:
traffic = pd.read_csv("datasets/traffic/DebitCirculation.csv")
print(traffic.columns.tolist())

['ide_sectn_trafc', 'num_sectn_trafc', 'des_debut_sous_route', 'des_fin_sous_route', 'rtss_debut_chaing', 'rtss_fin_chaing', 'annee_en_cours', 'annee2', 'annee3', 'annee4', 'annee5', 'annee6', 'annee7', 'annee8', 'annee9', 'annee10', 'dat_debut_sectn_trafc', 'rtss_debut', 'val_chang_debut', 'rtss_fin', 'val_chang_fin', 'djma_annee_1', 'val_djma_annee_1', 'djma_annee_2', 'val_djma_annee_2', 'djma_annee_3', 'val_djma_annee_3', 'djma_annee_4', 'val_djma_annee_4', 'djma_annee_5', 'val_djma_annee_5', 'djma_annee_6', 'val_djma_annee_6', 'djma_annee_7', 'val_djma_annee_7', 'djma_annee_8', 'val_djma_annee_8', 'djma_annee_9', 'val_djma_annee_9', 'djma_annee_10', 'val_djma_annee_10', 'djme_annee_1', 'val_djme_annee_1', 'djme_annee_2', 'val_djme_annee_2', 'djme_annee_3', 'val_djme_annee_3', 'djme_annee_4', 'val_djme_annee_4', 'djme_annee_5', 'val_djme_annee_5', 'djme_annee_6', 'val_djme_annee_6', 'djme_annee_7', 'val_djme_annee_7', 'djme_annee_8', 'val_djme_annee_8', 'djme_annee_9', 'val_djme_ann

In [41]:
# Look for Montreal-related entries
print(traffic[traffic['des_debut_sous_route'].str.contains('Montréal|Montreal', case=False, na=False)])

      ide_sectn_trafc num_sectn_trafc              des_debut_sous_route  \
1891            18033      0002016006  1re Avenue/boul Montréal-Toronto   
3267             8260      0031505000       R-148 Ouest, ch de Montréal   
5126            29737      0001305000             Boul. Gouin, Montréal   

                     des_fin_sous_route        rtss_debut_chaing  \
1891                         A-20 OUEST    00020-02-073-33B0 (0)   
3267               Bretelles A-50 Ouest    00315-01-005-000C (0)   
5126  Bretelles boulevard Samson, Laval  00013-02-050-000D (673)   

               rtss_fin_chaing  \
1891   00020-02-073-33B0 (627)   
3267  00315-01-005-000C (1930)   
5126   00013-02-064-000D (758)   

                                         annee_en_cours  \
1891        2024 DJMA: / DJME: / DJMH: / %cam: / 30e h:   
3267  2024 DJMA:11300 / DJME:12200 / DJMH:10100 / %c...   
5126  2024 DJMA:160000 / DJME:173000 / DJMH:144000 /...   

                                                 ann

## Weather

In [45]:
weather_2016 = pd.read_csv("datasets/weather/montreal_weather_2016.csv")
weather_2017 = pd.read_csv("datasets/weather/montreal_weather_2017.csv")
weather_2018 = pd.read_csv("datasets/weather/montreal_weather_2018.csv")
weather_2019 = pd.read_csv("datasets/weather/montreal_weather_2019.csv")
weather_2020 = pd.read_csv("datasets/weather/montreal_weather_2020.csv")
weather_2021 = pd.read_csv("datasets/weather/montreal_weather_2021.csv")
weather_2022 = pd.read_csv("datasets/weather/montreal_weather_2022.csv")
weather_2023 = pd.read_csv("datasets/weather/montreal_weather_2023.csv")
weather_2024 = pd.read_csv("datasets/weather/montreal_weather_2024.csv")
weather_2025 = pd.read_csv("datasets/weather/montreal_weather_2025.csv")

weather = pd.concat([weather_2016, weather_2017, weather_2018, weather_2019,
                     weather_2020, weather_2021, weather_2022, weather_2023, weather_2024, weather_2025], ignore_index=True)

In [46]:
keep_cols = [
    'Date/Time',
    'Max Temp (°C)',
    'Min Temp (°C)',
    'Mean Temp (°C)',
    'Total Precip (mm)',
    'Snow on Grnd (cm)'
]
weather = weather[keep_cols]
weather = weather.rename(columns={
    'Date/Time': 'Date',
    'Max Temp (°C)': 'MaxTemp',
    'Min Temp (°C)': 'MinTemp',
    'Mean Temp (°C)': 'MeanTemp',
    'Total Precip (mm)': 'Precip',
    'Snow on Grnd (cm)': 'SnowOnGround'
})

weather['Date'] = pd.to_datetime(weather['Date'])

print(weather.info())
print(f"Date range: {weather['Date'].min()} to {weather['Date'].max()}")

<class 'pandas.DataFrame'>
RangeIndex: 3653 entries, 0 to 3652
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   Date          3653 non-null   datetime64[us]
 1   MaxTemp       3604 non-null   float64       
 2   MinTemp       3613 non-null   float64       
 3   MeanTemp      3603 non-null   float64       
 4   Precip        3584 non-null   float64       
 5   SnowOnGround  1253 non-null   float64       
dtypes: datetime64[us](1), float64(5)
memory usage: 171.4 KB
None
Date range: 2016-01-01 00:00:00 to 2025-12-31 00:00:00


In [47]:
weather.to_csv("datasets/weather_cleaned.csv", index=False)